# Metaflora Incubus v1 — free GPU build
This notebook uses a compact <=7.5B QLoRA profile: 4-bit NF4, 2048-token training context and LoRA rank 16. A free T4 is not guaranteed to be available, and 9B training is deliberately rejected. Checkpoints go to Drive or a separate private Hub branch. Public upload remains blocked until the signed eval gates pass.

In [ ]:
import os
import shutil
import site
import subprocess
import sys
from pathlib import Path

trusted_code_revision = "9251b938eb44a38391fd7b3a69e9d03eb6542ce7"
repository = "/content/metaflora-incubus"
os.chdir("/content")
shutil.rmtree(repository, ignore_errors=True)
subprocess.run(["git", "init", repository], check=True)
subprocess.run(
    [
        "git",
        "-C",
        repository,
        "remote",
        "add",
        "origin",
        "https://github.com/metaflora-app/metaflora-incubus.git",
    ],
    check=True,
)
subprocess.run(
    ["git", "-C", repository, "fetch", "--depth", "1", "origin", trusted_code_revision],
    check=True,
)
subprocess.run(["git", "-C", repository, "checkout", "--detach", "FETCH_HEAD"], check=True)
checked_out = subprocess.run(
    ["git", "-C", repository, "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if checked_out != trusted_code_revision:
    raise RuntimeError("trusted code revision verification failed")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--require-hashes",
        "-r",
        "/content/metaflora-incubus/requirements/cloud-linux.lock",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "/content/metaflora-incubus"],
    check=True,
)
library_roots = sorted(
    {
        str(path)
        for root in site.getsitepackages()
        for path in Path(root).glob("nvidia/*/lib")
        if path.is_dir()
    }
)
if not library_roots:
    raise RuntimeError("CUDA runtime libraries were not installed")
library_path_parts = [*library_roots]
if os.environ.get("LD_LIBRARY_PATH"):
    library_path_parts.append(os.environ["LD_LIBRARY_PATH"])
runtime_environment = dict(os.environ)
runtime_environment["LD_LIBRARY_PATH"] = os.pathsep.join(library_path_parts)
os.environ["LD_LIBRARY_PATH"] = runtime_environment["LD_LIBRARY_PATH"]
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
runtime_environment["PYTORCH_CUDA_ALLOC_CONF"] = os.environ["PYTORCH_CUDA_ALLOC_CONF"]
subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch; import bitsandbytes; from peft import LoraConfig; from transformers import AutoModelForCausalLM; tensor = torch.ones((64, 64), device='cuda', dtype=torch.float16); bitsandbytes.functional.quantize_4bit(tensor, quant_type='nf4')",
    ],
    check=True,
    env=runtime_environment,
)
source_root = f"{repository}/src"
if source_root not in sys.path:
    sys.path.insert(0, source_root)
os.chdir("/content/metaflora-incubus")

In [ ]:
# Add the short private key named INCUBUS_BOOTSTRAP to Colab Secrets.
from pathlib import Path

from metaflora_incubus.cloud_bootstrap import decrypt_cloud_bootstrap, install_cloud_bootstrap

try:
    from google.colab import userdata
except ImportError:
    from kaggle_secrets import UserSecretsClient

    encoded_bootstrap = UserSecretsClient().get_secret("INCUBUS_BOOTSTRAP")
else:
    try:
        encoded_bootstrap = userdata.get("INCUBUS_BOOTSTRAP")
    except Exception as exc:
        raise RuntimeError(
            "Colab Secret INCUBUS_BOOTSTRAP is missing or notebook access is disabled"
        ) from exc
if not isinstance(encoded_bootstrap, str) or not encoded_bootstrap.strip():
    raise RuntimeError("INCUBUS_BOOTSTRAP is empty")

encrypted_bootstrap = Path("configs/cloud/bootstrap-v1.enc").read_bytes()
bootstrap_payload = decrypt_cloud_bootstrap(encrypted_bootstrap, encoded_bootstrap)
install_cloud_bootstrap(bootstrap_payload, home=Path.home(), environment=os.environ)
runtime_environment = dict(os.environ)

In [ ]:
# All private run values come from the single bootstrap secret.
command = [
    sys.executable,
    "scripts/run_free_gpu.py",
    "--execute",
    "--run-id",
    "incubus-v1-run",
    "--parameter-count",
    os.environ["INCUBUS_PARAMETER_COUNT"],
    "--checkpoint-backend",
    "hf_private_branch",
    "--checkpoint-location",
    "metaflora/incubus-checkpoints",
    "--checkpoint-branch",
    "incubus-training-v1",
]
subprocess.run(command, check=True, env=runtime_environment)